In [ ]:
import pandas as pd
from pathlib import Path
import json
from collections import defaultdict

In [20]:
ntbk_dir = Path.cwd().parent
data_root = ntbk_dir / "data"
participants_file_path = data_root / "TDBRAIN_participants_V3.xlsx"
part_df = pd.read_excel(participants_file_path)

In [ ]:
print("Dataset null count:", len(part_df[part_df["Dataset"].isna()]))
print("indication null count:", len(part_df[part_df["indication"].isna()]))
print("weight null count:", len(part_df[part_df["Weight (kg)"].isna()]))
print("Height null count:", len(part_df[part_df["Height (cm)"].isna()]))
print("sleep null count:", len(part_df[part_df["sleep"].isna()]))

Dataset null count: 1075
indication null count: 257
formal_status null count: 739
age null count: 0
gender null count: 0
education null count: 55
weight null count: 692
Height null count: 692


In [43]:
print("formal_status UNKNOWN count:", len(part_df[part_df["formal_status"] == "UNKNOWN"]))
print("age null count:", len(part_df[part_df["age"].isna()]))
print("gender null count:", len(part_df[part_df["gender"].isna()]))
print("education null count:", len(part_df[part_df["education"].isna()]))
print("n_oddb_CP null count:", len(part_df[part_df["n_oddb_CP"].isna()]))
print("n_oddb_FP null count:", len(part_df[part_df["n_oddb_FP"].isna()]))
print("n_oddb_CN null count:", len(part_df[part_df["n_oddb_CN"].isna()]))
print("n_oddb_FN null count:", len(part_df[part_df["n_oddb_FN"].isna()]))

formal_status UNKNOWN count: 739
age null count: 1
gender null count: 0
education null count: 55
n_oddb_CP null count: 136
n_oddb_FP null count: 136
n_oddb_CN null count: 136
n_oddb_FN null count: 136


In [76]:
neo_cols = [f'neoFFI_q{i}' for i in range(1, 61)]
filt_df = part_df[
    (part_df["formal_status"] != "UNKNOWN") &
    (part_df["age"].notna()) &
    (part_df["gender"].notna()) &
    (part_df["education"].notna()) &
    (part_df["n_oddb_CP"].notna()) &
    (part_df["n_oddb_FP"].notna()) &
    (part_df["n_oddb_CN"].notna()) &
    (part_df["n_oddb_FN"].notna()) &
    (part_df["avg_rt_oddb_CP"].notna()) &
    (part_df[neo_cols].notna().all(axis=1))
]
print("Subjects after filtering:", len(filt_df))

print("avg_rt_oddb_FP null count after filtering:", filt_df["avg_rt_oddb_FP"].isna().sum())

filt_df = filt_df[filt_df["nrSessions"] == 1]
print("Subjects after filtering:", len(filt_df))

Subjects after filtering: 375
avg_rt_oddb_FP null count after filtering: 194
Subjects after filtering: 345


In [77]:
filt_df["formal_status"].value_counts()

formal_status
MDD             155
ADHD             81
OCD              36
INSOMNIA         32
HEALTHY          21
TINNITUS          9
CHRONIC PAIN      7
ADD               3
TINNITUS/MDD      1
Name: count, dtype: int64

In [ ]:
# checking for whether all filtered dataframe rows exist within the set of bdf files
subj_ids = set()
bdf_path_map = {}
for path in Path(data_root).rglob("*.bdf"):
    print(path)
    subj_id = path.name.split("_")[0]
    subj_ids.add(subj_id)
    if bdf_path_map.get(subj_id) is None:

filt_df["TDBRAIN_ID"].isin(subj_ids)

/Users/swaraagsistla/Documents/ComputerScience/CollegeResearch/NeuralEval/neural-privacy-evals/data/TDBRAIN_Dataset_V3_1/sub-88010617/ses-1/eeg/sub-88010617_ses-1_task-restEO_eeg.bdf
/Users/swaraagsistla/Documents/ComputerScience/CollegeResearch/NeuralEval/neural-privacy-evals/data/TDBRAIN_Dataset_V3_1/sub-88010617/ses-1/eeg/sub-88010617_ses-1_task-restEC_eeg.bdf
/Users/swaraagsistla/Documents/ComputerScience/CollegeResearch/NeuralEval/neural-privacy-evals/data/TDBRAIN_Dataset_V3_1/sub-88045085/ses-1/eeg/sub-88045085_ses-1_task-restEC_eeg.bdf
/Users/swaraagsistla/Documents/ComputerScience/CollegeResearch/NeuralEval/neural-privacy-evals/data/TDBRAIN_Dataset_V3_1/sub-88045085/ses-1/eeg/sub-88045085_ses-1_task-restEO_eeg.bdf
/Users/swaraagsistla/Documents/ComputerScience/CollegeResearch/NeuralEval/neural-privacy-evals/data/TDBRAIN_Dataset_V3_1/sub-88041617/ses-1/eeg/sub-88041617_ses-1_task-restEO_eeg.bdf
/Users/swaraagsistla/Documents/ComputerScience/CollegeResearch/NeuralEval/neural-priv

281     True
312     True
314     True
315     True
316     True
        ... 
1358    True
1359    True
1360    True
1361    True
1362    True
Name: TDBRAIN_ID, Length: 345, dtype: bool

In [ ]:
bdf_path_map = {}

for subj_id in subj_ids:
    if bdf_path_map.get(subj_id) is None:
        bdf_path_map

for row in filt_df.iterrows():
    if bdf_path_map.get(row["TDBRAIN_ID"]) is None:
        bdf_path_map[row["TDBRAIN_ID"]] = []

In [79]:
filt_df.head()

,TDBRAIN_ID,DISC/REP,indication,formal_status,Dataset,Consent,sessSeason,sessTime,Responder,Remitter,...,BDI_pre,BDI_post,rTMS PROTOCOL,ADHD_pre_Hyp_leading,ADHD_pre_Att_leading,ADHD_post_Att_leading,ADHD_post_Hyp_leading,NF Protocol,YBOCS_pre,YBOCS_post
281,sub-87980373,DISCOVERY,HEALTHY,HEALTHY,NaN,YES,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
312,sub-87984605,DISCOVERY,CHRONIC PAIN,CHRONIC PAIN,NaN,YES,summer,morning,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
314,sub-87987485,DISCOVERY,CHRONIC PAIN,CHRONIC PAIN,NaN,YES,summer,morning,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
315,sub-87991085,DISCOVERY,CHRONIC PAIN,CHRONIC PAIN,NaN,YES,fall,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
316,sub-87991401,DISCOVERY,CHRONIC PAIN,CHRONIC PAIN,NaN,YES,fall,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
